# Classification: Baseline → Best — Will This Order Get Cancelled? 📦❌

Same Online Retail II shop as Days 2 and 5. This time, a new question:

**"Looking at an order the moment it's placed — before we know what happens next — can we guess whether it'll end up cancelled?"**

We'll build 3 models, each one better than the last, the same way a shop might slowly get better at spotting a risky order before it ships:

1. A simple guesser (Logistic Regression)
2. A better guesser (Decision Tree)
3. Our best guesser (LightGBM + SMOTE — the same combo used in the RetainIQ project)

**Spoiler, because it's an important lesson:** along the way we'll find a sneaky bug that made an early version of this "perfect" — and perfect is never something to celebrate without checking why.

## 0. What does "cancelled" even mean here?

**Analogy:** imagine every order slip in the shop. If a slip's order number starts with the letter "C", it means "this order got cancelled/returned" — like a big red C stamped on the slip. About **15 out of every 100 orders** get that stamp. Our job: build a system that looks at an order BEFORE anyone stamps it, and guesses whether it's headed for that stamp.

In [ ]:
import pandas as pd
import numpy as np
from features import build_order_features

df = pd.read_csv('data/raw/online_retail_II.csv', encoding='latin1')
orders = build_order_features(df)
print(f"Total orders: {len(orders):,}")
print(f"Cancelled orders: {orders['IsCancelled'].sum():,}")
print(f"Cancellation rate: {orders['IsCancelled'].mean():.1%}")
orders.head()

## 1. Why this is called an "imbalanced" problem

**Analogy:** imagine sorting 100 marbles, and only 15 are red (cancelled), 85 are blue (normal). If a lazy robot just guessed "blue" for every single marble, it would be right 85% of the time — sounds impressive, but it would NEVER catch a single red marble. That's exactly the trap with imbalanced data, and it's the whole reason this project exists instead of a simple 50/50 problem.

In [ ]:
import matplotlib.pyplot as plt

orders['IsCancelled'].value_counts().plot(kind='bar', color=['#4c72b0','#c44e52'])
plt.xticks([0,1], ['Not Cancelled', 'Cancelled'], rotation=0)
plt.title('Class balance — this is what "imbalanced" looks like')
plt.ylabel('Number of orders')
plt.show()

## 2. Building the features — what would a shop actually know at order time?

**Analogy:** think about what a shop clerk could see the MOMENT an order is placed — how many different items, how much stuff total, how expensive on average, what country it's shipping to, what day/hour it is. That's all fair game. What they CAN'T know yet is whether it'll be cancelled — that happens later, if at all.

In [ ]:
orders[['num_unique_products','total_quantity','total_value','avg_unit_price','Country','DayOfWeek','Hour']].head()

## 3. A bug I actually hit while building this — and why it matters

The first version of `total_quantity` and `total_value` just summed up `Quantity × Price` directly from the raw data. That seemed reasonable... until every single model scored **100% accuracy**. That should never make you happy — it should make you suspicious.

**Analogy:** imagine grading a test where the answer key was accidentally printed on the back of every question. A student getting 100% doesn't prove they're a genius — it proves the answer was leaking through. That's exactly what happened here.

**What was actually leaking:** in this dataset, a cancelled order's `Quantity` is recorded as a **negative number** — that's literally how a cancellation is logged in this particular export. So `total_quantity < 0` was basically just... the label, wearing a disguise. The model wasn't learning "which orders are risky" — it was learning "which orders already got stamped C", which we're not allowed to know at prediction time.

In [ ]:
# Proof of the leak: 100% of cancelled line items have negative Quantity
df['IsCancelled'] = df['Invoice'].astype(str).str.startswith('C')
leak_check = (df[df['IsCancelled']]['Quantity'] < 0).mean()
print(f"Fraction of cancelled line items with negative Quantity: {leak_check:.0%}")
print("That's not a pattern to learn from — that's the answer key.")

**The fix:** `features.py` now uses `.abs()` (absolute value) on quantity before summing — so the features describe the **size** of the order (how much stuff, how much it's worth), not its sign. The sign was the leak; the size is the real signal. This is why `features.py` is a separate, documented file instead of scratch code buried in the notebook — the fix and the reasoning behind it live in one place, not lost in a rerun cell.

## 4. Splitting the data — and why `stratify` matters here

**Analogy:** if you're taste-testing a bag of 85 blue and 15 red marbles by grabbing two random handfuls, you could easily get unlucky and end up with a test handful that has almost no red marbles at all — then you'd have no real way to check if your model can spot red ones. `stratify=y` tells the split "keep the same 85/15 mix in both handfuls" — so the test set is a fair, representative check.

In [ ]:
from sklearn.model_selection import train_test_split

top_countries = orders['Country'].value_counts().head(10).index
orders['Country'] = orders['Country'].where(orders['Country'].isin(top_countries), 'Other')

feature_cols_num = ['num_unique_products','total_quantity','total_value','avg_unit_price','Hour']
feature_cols_cat = ['Country','DayOfWeek']

X = orders[feature_cols_num + feature_cols_cat]
y = orders['IsCancelled'].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Train: {X_train.shape[0]:,} orders  |  Test: {X_test.shape[0]:,} orders")
print(f"Train cancel rate: {y_train.mean():.1%}  |  Test cancel rate: {y_test.mean():.1%}")

In [ ]:
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer

# Numbers get scaled (put on a comparable range), categories get one-hot encoded
# (turned into 0/1 columns) — standard prep so every model sees clean, comparable inputs.
preprocess = ColumnTransformer([
    ('num', StandardScaler(), feature_cols_num),
    ('cat', OneHotEncoder(handle_unknown='ignore'), feature_cols_cat),
])

## 5. Model 1 — Logistic Regression (the simple guesser)

**Analogy:** this is like a shop clerk who's only allowed to draw ONE straight line through all their past experience and say "everything on this side is risky, everything on that side isn't." Simple, fast, and a fair starting point — but real risk rarely follows one straight line.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, confusion_matrix

logreg_pipe = Pipeline([('prep', preprocess), ('clf', LogisticRegression(max_iter=1000))])
logreg_pipe.fit(X_train, y_train)
pred_logreg = logreg_pipe.predict(X_test)

print(classification_report(y_test, pred_logreg, target_names=['Not Cancelled','Cancelled'], digits=3))

**Read this carefully:** overall accuracy is 86% — sounds fine! But look at the `Cancelled` row's **recall: 0.087**. That means this model catches less than **9 out of every 100 actual cancellations.** This is the marble trap from Section 1, playing out for real. Accuracy alone would have completely hidden this.

In [ ]:
cm1 = confusion_matrix(y_test, pred_logreg)
print("Confusion matrix:")
print(f"                  Predicted: Not Cancelled   Predicted: Cancelled")
print(f"Actually Not Cancelled:      {cm1[0][0]:>18}   {cm1[0][1]:>18}")
print(f"Actually Cancelled:          {cm1[1][0]:>18}   {cm1[1][1]:>18}")
print()
print(f"Of {cm1[1].sum()} orders that WERE actually cancelled, this model only caught {cm1[1][1]}.")

## 6. What a confusion matrix's 4 boxes actually mean (in plain words)

**Analogy — a smoke detector:**
- **Top-left (True Negative):** no fire, no alarm. Correct silence. ✅
- **Bottom-right (True Positive):** fire, alarm goes off. Correct catch. ✅
- **Top-right (False Positive):** no fire, alarm goes off anyway. Annoying, but safe — a false alarm. ⚠️
- **Bottom-left (False Negative):** fire, but NO alarm. The dangerous one — a real risk, completely missed. 🚨

For our cancellation problem, a **False Negative** (an order that WILL be cancelled, but we predicted "fine") is the box that actually costs the business money — they ship something that comes right back. That's the box we most want to shrink.

## 7. Model 2 — Decision Tree (a smarter guesser)

**Analogy:** instead of one straight line, a decision tree is like a flowchart of yes/no questions — "is total_quantity over 50? if yes, is avg_unit_price under £2? ..." — branching until it reaches an answer. This lets it capture patterns a single straight line can't.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

tree_pipe = Pipeline([('prep', preprocess), ('clf', DecisionTreeClassifier(max_depth=6, random_state=42))])
tree_pipe.fit(X_train, y_train)
pred_tree = tree_pipe.predict(X_test)

print(classification_report(y_test, pred_tree, target_names=['Not Cancelled','Cancelled'], digits=3))

**Progress:** Cancelled-class recall jumped from **0.087 → 0.801** — this model now catches roughly 8 out of every 10 actual cancellations, a massive improvement over the straight-line model. Accuracy actually looks slightly lower than before (91.5% vs 86%) — proof that accuracy alone was never the number that mattered here.

## 8. Model 3 — LightGBM + SMOTE (the best guesser, same combo as RetainIQ)

Two upgrades happen together here:

**LightGBM** is like having a whole committee of decision trees vote together, instead of trusting just one — each tree in the committee focuses extra hard on the mistakes the trees before it made.

**SMOTE — the piggy bank analogy:** imagine you only have 15 examples of "risky orders" to learn from, versus 85 examples of "normal orders." SMOTE doesn't just copy-paste those 15 examples to pad the numbers (that would just be memorizing the same 15 cases harder) — it looks at real risky orders that are similar to each other and invents new, realistic in-between examples, like plotting new points on a line connecting two real ones. Now the model gets a fairer, more balanced set to learn from.

In [ ]:
from imblearn.over_sampling import SMOTE
import lightgbm as lgb

X_train_prep = preprocess.fit_transform(X_train)
X_test_prep = preprocess.transform(X_test)

print("Before SMOTE:", dict(y_train.value_counts()))
smote = SMOTE(random_state=42)
X_train_sm, y_train_sm = smote.fit_resample(X_train_prep, y_train)
print("After SMOTE: ", dict(pd.Series(y_train_sm).value_counts()))

In [ ]:
lgbm = lgb.LGBMClassifier(random_state=42, verbose=-1)
lgbm.fit(X_train_sm, y_train_sm)
pred_lgbm = lgbm.predict(X_test_prep)

print(classification_report(y_test, pred_lgbm, target_names=['Not Cancelled','Cancelled'], digits=3))

**Best result:** Cancelled-class recall climbs again to **0.866** — this model now catches roughly 87 out of every 100 actual cancellations, up from 80 for the plain tree and just 9 for the baseline. Precision dipped slightly (0.656) — meaning more false alarms — which is a genuine tradeoff, not a free win. Whether that tradeoff is worth it depends on the real business cost of a false alarm vs. a missed cancellation.

## 9. The full comparison, side by side

In [ ]:
comparison = pd.DataFrame({
    'Model': ['Logistic Regression (baseline)', 'Decision Tree', 'LightGBM + SMOTE'],
    'Accuracy': [0.859, 0.915, 0.913],
    'Cancelled Precision': [0.691, 0.683, 0.656],
    'Cancelled Recall': [0.087, 0.801, 0.866],
    'Cancelled F1': [0.155, 0.737, 0.746],
})
comparison

In [ ]:
fig, ax = plt.subplots(figsize=(8,5))
comparison.set_index('Model')[['Cancelled Precision','Cancelled Recall','Cancelled F1']].plot(kind='bar', ax=ax)
ax.set_title('The metric that actually matters: performance on the Cancelled class')
ax.set_ylabel('Score')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.show()

**The story this table tells:** accuracy barely moved (86% → 91%), but that number was always hiding the real picture. Recall on the class we actually care about went from catching **9 out of 100** cancellations to catching **87 out of 100** — that's the entire point of not trusting accuracy alone on imbalanced data.

## 10. What actually drove the predictions?

**Analogy:** if the model is a chef, feature importance is like asking "which ingredient did you rely on most to guess the final dish?" 

In [ ]:
feat_names = feature_cols_num + list(
    preprocess.named_transformers_['cat'].get_feature_names_out(feature_cols_cat)
)
importances = pd.Series(lgbm.feature_importances_, index=feat_names).sort_values(ascending=False)

fig, ax = plt.subplots(figsize=(8,5))
importances.head(8).plot(kind='barh', ax=ax, color='#55a868')
ax.invert_yaxis()
ax.set_title('Top 8 features the model actually relied on')
plt.tight_layout()
plt.show()

**Takeaway:** `num_unique_products`, `total_quantity`, and `total_value` dominate — in plain words, **how big and varied an order is** matters more to cancellation risk than *which country* it's shipping to or *what day* it was placed. That's a genuinely useful, explainable finding a business could act on — e.g., double-checking unusually large or varied orders before shipping.

## Summary

| Step | What happened |
|---|---|
| Framing | Predict cancellation using only info known at order time — no peeking at the label |
| The trap | Accuracy alone hid a model that caught almost no real cancellations |
| The bug | Signed Quantity secretly leaked the answer; fixed with `.abs()` in `features.py` |
| Baseline → Best | Logistic Regression (9% recall) → Decision Tree (80% recall) → LightGBM+SMOTE (87% recall) |
| What matters most | Order size (unique products, quantity, value) — not country or day |

**What I'd do next:** try adjusting the LightGBM decision threshold (currently a straight 50/50 cutoff) to trade precision and recall deliberately, based on what a false alarm vs. a missed cancellation actually costs a real business — rather than accepting scikit-learn's default cutoff as if it were the "right" one.
